# Table S7: go_depth / standard_go_size / standard_gene_size hyperparameter search

In [ ]:
#!/usr/bin/env python3
"""
Compute weighted scib scores across ablation folders.
Score = bio_conservation * 0.6 + batch_correction * 0.4
"""

import os
import pandas as pd

BASE_DIR = os.path.abspath(
    "../.experiment_data/primary/"
    "sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_"
    "scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_"
    "ENCFF173VDJ_gaf_standard_with_rna_s0_ablation"
)

BIO_WEIGHT = 0.6
BATCH_WEIGHT = 0.4


def extract_aggregate_score(csv_path):
    """
    Load a scib results CSV and extract the aggregate score value.
    The CSV has two data rows: X_emb (values) and Metric Type (labels).
    The aggregate score column is the one whose Metric Type row == 'Aggregate score'.
    Returns the float value from that column in the X_emb row.
    """
    df = pd.read_csv(csv_path, index_col=0)
    # Row "Metric Type" contains column labels; row "X_emb" contains values
    metric_type_row = df.loc["Metric Type"]
    value_row = df.loc["X_emb"]
    # Find the column where Metric Type == 'Aggregate score'
    agg_cols = metric_type_row[metric_type_row == "Aggregate score"].index.tolist()
    if not agg_cols:
        raise ValueError(f"No 'Aggregate score' column found in {csv_path}")
    return float(value_row[agg_cols[0]])


results = []

# Collect all ablation subdirectories
folders = sorted([
    d for d in os.listdir(BASE_DIR)
    if os.path.isdir(os.path.join(BASE_DIR, d)) and d.startswith("ablation_")
])

for folder in folders:
    folder_path = os.path.join(BASE_DIR, folder)
    label_csv = os.path.join(folder_path, "scib_results_label.csv")
    site_csv = os.path.join(folder_path, "scib_results_Site.csv")

    # Check files exist
    missing = [p for p in [label_csv, site_csv] if not os.path.isfile(p)]
    if missing:
        print(f"[ERROR] {folder}: missing files: {[os.path.basename(p) for p in missing]}")
        continue

    try:
        bio_score = extract_aggregate_score(label_csv)
        batch_score = extract_aggregate_score(site_csv)
        weighted = bio_score * BIO_WEIGHT + batch_score * BATCH_WEIGHT

        results.append({
            "Folder": folder,
            "Bio conservation": bio_score,
            "Batch correction": batch_score,
            f"Weighted score ({BIO_WEIGHT}*bio + {BATCH_WEIGHT}*batch)": weighted,
        })

    except Exception as e:
        print(f"[ERROR] {folder}: {e}")

# Build and display results table
if results:
    df_results = pd.DataFrame(results).set_index("Folder")
    df_results = df_results.sort_index()

    pd.set_option("display.float_format", "{:.4f}".format)
    pd.set_option("display.max_colwidth", None)
    pd.set_option("display.width", 120)

    print("\n=== Ablation Scores ===\n")
    print(df_results.to_string())

    # Save to CSV
    out_csv = os.path.join(BASE_DIR, "ablation_weighted_scores.csv")
    df_results.to_csv(out_csv)
    print(f"\nResults saved to: {out_csv}")
else:
    print("\nNo results could be computed.")

In [ ]:
df_results